# Tiny LLM Lab — Colab end-to-end training

Notebook này chạy toàn bộ pipeline bằng Python API, không gọi `train_architectures.py` hay CLI training. Mục tiêu là nhìn thấy từng bước: data contract → tokenizer/token artifacts → batch loader → model → loss → optimizer → training → checkpoint → evaluation → KV-cache benchmark.

Mặc định notebook train một model GQA khoảng 45–50M parameters trên TinyStories 100M train tokens, context 512 và global batch 32,768 tokens/update. Sau khi smoke test GQA, có thể đổi `ARCHITECTURE` sang `mha`, `mla`, `moe` hoặc `v4` ở cell model.

## 0. Quy ước trước khi chạy

- Hãy chạy các cell theo thứ tự từ trên xuống.
- Dataset và checkpoint được lưu trên Google Drive để không mất khi Colab reset.
- Nếu đã có `tinystories_100m.manifest.json`, notebook sẽ dùng lại artifact đó. Nếu chưa có, cell dataset sẽ tải TinyStories streaming và tạo artifact mới.
- Hãy chạy cell model và cell sanity check trước khi train đủ 100M tokens; nếu OOM, giảm `BATCH_SIZE` xuống 4 và tăng gradient accumulation lên 16.

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    # Colab: sửa path nếu repo nằm ở vị trí khác trên Drive.
    PROJECT_ROOT = Path('/content/drive/MyDrive/tiny-llm-lab/llm-lab')
except ModuleNotFoundError:
    # Local smoke test: chạy notebook từ project root.
    PROJECT_ROOT = Path.cwd()
assert (PROJECT_ROOT / 'src' / 'train.py').exists(), (
    f'Không tìm thấy project tại {PROJECT_ROOT}. Hãy upload/clone repo vào Drive trước.'
)
print(PROJECT_ROOT)

## 1. Cài dependency và import source code

Colab đã có PyTorch. Cell dưới chỉ cài các package data/tokenizer cần thiết; đây không phải cách chạy training CLI.

In [ ]:
%pip install -q numpy tokenizers tiktoken datasets matplotlib tqdm

import sys
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import json
from contextlib import nullcontext
import math
import random
import time
from tqdm.auto import tqdm
from dataclasses import asdict

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

from benchmarking.compute import (
    active_parameter_count,
    collect_kv_cache_stats,
    estimate_flops,
    estimate_kv_cache,
)
from benchmarking.inference import run_generation, synchronize
from data import load_token_artifacts
from data.datasets import NextTokenDataset, StatefulBatchLoader, take_token_budget
from data.manifest import build_manifest
from data.splits import split_documents_three
from data.tokenizer import build_tokenizer
from evaluation.loss import evaluate_loss_stats
from config import TrainingConfig
from training.loop import TrainingState, restore_rng_state
from visualization import save_training_plots
from training.optim import build_adamw
from models.registry import build_model, model_metadata, parameter_count

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__)
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Chọn experiment và seed

Mọi architecture phải dùng cùng các giá trị này nếu muốn so sánh công bằng. MoE là ngoại lệ về **total resident parameters**, nhưng active parameters/FLOPs vẫn được report riêng.

In [ ]:
SEED = 42
ARCHITECTURE = 'gqa'  # đổi thành: mha, gqa, mla, moe, v4
TRAIN_TOKENS = 100_000_000
VALIDATION_TOKENS = 5_000_000
TEST_TOKENS = 5_000_000
MAX_EXAMPLES = 1_000_000
CONTEXT_LENGTH = 512
BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 8
EVAL_EVERY_STEPS = 250
EVAL_BATCHES_DURING_TRAINING = 20
SAVE_EVERY_STEPS = 1500  # checkpoint trung gian thưa hơn; vẫn giữ best + final
ARTIFACT_NAME = 'tinystories_100m'  # artifact scale-up; không dùng lại pilot 10M
RESUME_CHECKPOINT = None  # hoặc Path('/.../checkpoint.pt') để resume

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_DIR = PROJECT_ROOT / 'data'
RUN_DIR = PROJECT_ROOT / 'runs' / f'colab_{ARCHITECTURE}_50m_100m_notebook'
DATA_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Run directory:', RUN_DIR)

## 3. Tạo hoặc load fixed-token artifact

Tại sao cần bước này? Nếu mỗi model tự fit tokenizer và encode lại raw JSONL, số token thực tế có thể khác nhau. Artifact cố định giúp mọi model nhìn đúng cùng token stream.

Artifact gồm:

```text
train.npy
validation.npy
test.npy
tokenizer.json
manifest.json
```

In [ ]:
ARTIFACT_BASE = DATA_DIR / ARTIFACT_NAME
MANIFEST_PATH = ARTIFACT_BASE.with_suffix('.manifest.json')

if not MANIFEST_PATH.exists():
    from datasets import load_dataset

    print('Chưa có artifact — tải TinyStories streaming...')
    stream = load_dataset('roneneldan/TinyStories', split='train', streaming=True)
    documents = []
    for row in stream:
        text = str(row['text']).strip()
        if text:
            documents.append(text)
        if len(documents) >= MAX_EXAMPLES:
            break
    print('Documents:', len(documents))

    train_docs, validation_docs, test_docs = split_documents_three(
        documents, train_fraction=0.9, validation_fraction=0.05, seed=SEED
    )
    tokenizer = build_tokenizer('bpe', train_docs, vocab_size=16_000, min_frequency=2)
    train_full = tokenizer.encode_documents(train_docs)
    validation_full = tokenizer.encode_documents(validation_docs)
    test_full = tokenizer.encode_documents(test_docs)
    train_ids = np.asarray(take_token_budget(train_full, TRAIN_TOKENS), dtype=np.uint32)
    validation_ids = np.asarray(take_token_budget(validation_full, VALIDATION_TOKENS), dtype=np.uint32)
    test_ids = np.asarray(take_token_budget(test_full, TEST_TOKENS), dtype=np.uint32)

    tokenizer_path = ARTIFACT_BASE.with_suffix('.tokenizer.json')
    train_path = ARTIFACT_BASE.with_suffix('.train.npy')
    validation_path = ARTIFACT_BASE.with_suffix('.validation.npy')
    test_path = ARTIFACT_BASE.with_suffix('.test.npy')
    tokenizer.save(tokenizer_path)
    np.save(train_path, train_ids)
    np.save(validation_path, validation_ids)
    np.save(test_path, test_ids)

    import hashlib
    def token_sha256(values):
        raw = np.ascontiguousarray(values).view(np.uint8)
        return hashlib.sha256(raw).hexdigest()

    tokenizer_hash = hashlib.sha256(tokenizer_path.read_bytes()).hexdigest()
    source = {
        'id': 'roneneldan/TinyStories',
        'url': 'https://huggingface.co/datasets/roneneldan/TinyStories',
        'license': 'CDLA-Sharing-1.0',
        'license_url': 'https://cdla.dev/sharing-1-0/',
    }
    manifest = build_manifest(
        documents, train_docs, validation_docs, source, SEED, 0.9, test_docs,
        tokenizer_kind='byte_level_bpe',
        validation_fraction=0.05,
        tokenizer_vocab_size=tokenizer.vocab_size,
        tokenizer_sha256=tokenizer_hash,
        train_token_count=len(train_ids),
        validation_token_count=len(validation_ids),
        test_token_count=len(test_ids),
        target_train_tokens=TRAIN_TOKENS,
        target_validation_tokens=VALIDATION_TOKENS,
        target_test_tokens=TEST_TOKENS,
        token_id_dtype='uint32',
        train_token_sha256=token_sha256(train_ids),
        validation_token_sha256=token_sha256(validation_ids),
        test_token_sha256=token_sha256(test_ids),
    ).to_dict()
    manifest['data_file'] = str(ARTIFACT_BASE.with_suffix('.jsonl'))
    manifest['token_artifacts'] = {
        'train': str(train_path),
        'validation': str(validation_path),
        'test': str(test_path),
        'tokenizer': str(tokenizer_path),
    }
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    print('Đã tạo artifact:', MANIFEST_PATH)
else:
    print('Dùng artifact có sẵn:', MANIFEST_PATH)

In [ ]:
artifacts = load_token_artifacts(MANIFEST_PATH)
train_ids = artifacts.train_tokens
validation_ids = artifacts.validation_tokens
test_ids = artifacts.test_tokens
tokenizer = artifacts.tokenizer

print('Tokenizer vocab:', tokenizer.vocab_size)
print('Train tokens:', len(train_ids))
print('Validation tokens:', len(validation_ids))
print('Test tokens:', len(test_ids))
print('Contract:')
print(json.dumps(artifacts.contract, indent=2))

## 4. Biến token stream thành next-token batches

Với context `L`, một sample có dạng:

```text
input : t0 t1 t2 ... t(L-1)
target: t1 t2 t3 ... tL
```

`drop_last=True` cho train để mỗi optimizer update có đúng số token. Validation/test giữ batch cuối để final evaluation không bỏ token.

In [ ]:
train_dataset = NextTokenDataset(train_ids, CONTEXT_LENGTH, stride=CONTEXT_LENGTH)
validation_dataset = NextTokenDataset(validation_ids, CONTEXT_LENGTH, stride=CONTEXT_LENGTH)
test_dataset = NextTokenDataset(test_ids, CONTEXT_LENGTH, stride=CONTEXT_LENGTH)

train_loader = StatefulBatchLoader(
    train_dataset, BATCH_SIZE, shuffle=True, seed=SEED, drop_last=True
)
validation_loader = StatefulBatchLoader(
    validation_dataset, BATCH_SIZE, shuffle=False, seed=SEED, drop_last=False
)
test_loader = StatefulBatchLoader(
    test_dataset, BATCH_SIZE, shuffle=False, seed=SEED, drop_last=False
)

TOKENS_PER_UPDATE = BATCH_SIZE * CONTEXT_LENGTH * GRADIENT_ACCUMULATION_STEPS
MAX_STEPS = TRAIN_TOKENS // TOKENS_PER_UPDATE
TOKENS_PER_EPOCH = train_loader.tokens_per_epoch
print('Train windows:', len(train_dataset))
print('Tokens per optimizer update:', TOKENS_PER_UPDATE)
print('Optimizer steps for ~100M tokens:', MAX_STEPS)
print('Tokens per loader epoch:', TOKENS_PER_EPOCH)

## 5. Build model

Đây là chỗ thay architecture. Các model cùng nhận `input_ids` và trả logits `(batch, sequence, vocab)`. `use_cache=False` trong pretraining vì KV cache chỉ dành cho autoregressive inference.

In [ ]:
base_model_config = {
    'vocab_size': tokenizer.vocab_size,
    'context_length': CONTEXT_LENGTH,
    'emb_dim': 512,
    'n_heads': 8,
    'n_layers': 12,
    'hidden_dim': 1728,
    'drop_rate': 0.0,
    'dropout': 0.0,
    'qkv_bias': False,
    'tie_embeddings': True,
    'n_kv_groups': 2,
    'latent_dim': 128,
    'head_dim': 64,
    'q_lora_rank': 64,
    'rope_dim': 32,
    'rope_base': 10_000.0,
    'window_size': 128,
    'compress_ratios': [0, 2, 0, 2, 0, 2, 0, 2, 0, 2, 0, 2],
    'index_topk': 8,
    'moe_hidden_dim': 384,
    'shared_hidden_dim': 384,
    'num_experts': 4,
    'num_experts_per_tok': 1,
    'shared_expert_hidden_dim': 512,
    'norm_eps': 1e-6,
}

architecture_overrides = {
    'mha': {'hidden_dim': 1536},
    'mla': {'latent_dim': 128, 'q_lora_rank': 64, 'num_experts': 0, 'num_experts_per_tok': 0},
    'moe': {'hidden_dim': 384, 'num_experts': 4, 'num_experts_per_tok': 1, 'moe_hidden_dim': 384, 'shared_expert_hidden_dim': 256},
    'v4': {'compress_ratios': [0, 2, 0, 2, 0, 2, 0, 2, 0, 2, 0, 2], 'q_lora_rank': 64, 'latent_dim': 128, 'moe_hidden_dim': 384, 'shared_hidden_dim': 384},
}
model_config = {**base_model_config, **architecture_overrides.get(ARCHITECTURE, {})}
model = build_model(ARCHITECTURE, model_config).to(device)
metadata = model_metadata(ARCHITECTURE, model)
flops = estimate_flops(model, ARCHITECTURE)
kv = estimate_kv_cache(model, ARCHITECTURE, CONTEXT_LENGTH)
print(json.dumps({
    'architecture': ARCHITECTURE,
    'total_parameters': parameter_count(model),
    'active_parameters': active_parameter_count(model, ARCHITECTURE),
    'forward_flops_per_token': flops.forward_flops_per_token,
    'training_flops_per_token': flops.training_flops_per_token,
    'kv_cache_bytes_per_token': kv.bytes_per_token,
}, indent=2))

In [ ]:
# Sanity check: một forward pass trước khi train.
inputs, targets = next(train_loader.evaluation_iterator(max_batches=1))
inputs = inputs.to(device)
targets = targets.to(device)
with torch.no_grad():
    logits = model(inputs, use_cache=False)
    initial_loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
print('inputs:', tuple(inputs.shape))
print('logits:', tuple(logits.shape))
print('initial loss:', float(initial_loss))

## 6. Optimizer, scheduler và mixed precision

Mỗi microbatch tạo gradient. Sau `GRADIENT_ACCUMULATION_STEPS` microbatch mới gọi `optimizer.step()`. Vì vậy effective token budget/update lớn hơn batch vật lý.

In [ ]:
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
WARMUP_STEPS = min(max(1, min(150, MAX_STEPS // 10)), max(0, MAX_STEPS - 1))
MIN_LR_RATIO = 0.1
GRAD_CLIP_NORM = 1.0

training_config = TrainingConfig(
    batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    precision='auto',
    learning_rate=LEARNING_RATE,
    beta1=0.9,
    beta2=0.95,
    adam_eps=1e-8,
    weight_decay=WEIGHT_DECAY,
    max_steps=MAX_STEPS,
    warmup_steps=WARMUP_STEPS,
    min_lr_ratio=MIN_LR_RATIO,
    eval_every=EVAL_EVERY_STEPS,
    eval_batches=EVAL_BATCHES_DURING_TRAINING,
    grad_clip_norm=GRAD_CLIP_NORM,
    seed=SEED,
    save_every=SAVE_EVERY_STEPS,
)
optimizer = build_adamw(model, training_config)

from training.schedule import cosine_lr, set_optimizer_lr
use_amp = device.type == 'cuda'
amp_dtype = torch.bfloat16 if use_amp and torch.cuda.is_bf16_supported() else torch.float16
use_scaler = use_amp and amp_dtype == torch.float16
scaler = torch.amp.GradScaler('cuda', enabled=use_scaler)
print('AMP:', use_amp, amp_dtype if use_amp else 'disabled')

## 7. Evaluation và checkpoint helpers

Training evaluation chỉ lấy một số batch để monitor nhanh. Final validation/test bên dưới sẽ chạy toàn bộ held-out stream. Loss được tính bằng tổng negative log-likelihood chia cho tổng target tokens, không phải trung bình ngây thơ của batch losses.

In [ ]:
history = []
tokens_seen = 0
start_step = 0
best_validation_loss = float('inf')
best_validation_step = None
best_model_path = RUN_DIR / 'best_model.pt'
training_state = TrainingState(tokens_per_epoch=TOKENS_PER_EPOCH)

def evaluate_loader(loader, max_batches=None):
    return evaluate_loss_stats(model, loader, device, max_batches)

if RESUME_CHECKPOINT is not None:
    resume = torch.load(RESUME_CHECKPOINT, map_location=device, weights_only=False)
    if resume['architecture'] != ARCHITECTURE:
        raise ValueError('Resume architecture does not match ARCHITECTURE')
    if resume.get('data_contract') != artifacts.contract:
        raise ValueError('Resume artifact contract does not match current artifact')
    if any(key not in resume for key in ('loader_state', 'rng_state', 'training_config')):
        raise ValueError('Resume checkpoint is incomplete; use a checkpoint created by this updated notebook/runner')
    if resume.get('model_config') != model_config:
        raise ValueError('Resume model_config does not match the current notebook configuration')
    model.load_state_dict(resume['model_state'])
    optimizer.load_state_dict(resume['optimizer_state'])
    if resume.get('scaler_state'):
        scaler.load_state_dict(resume['scaler_state'])
    start_step = int(resume.get('global_step', resume['step']))
    tokens_seen = int(resume.get('global_train_tokens_seen', resume.get('tokens_seen', 0)))
    history = list(resume.get('history', []))
    best_validation_loss = min((float(row['validation_loss']) for row in history), default=float('inf'))
    best_validation_step = max((int(row['step']) for row in history if float(row['validation_loss']) == best_validation_loss), default=None)
    if resume.get('loader_state'):
        train_loader.load_state_dict(resume['loader_state']['train'])
        validation_loader.load_state_dict(resume['loader_state']['validation'])
        test_loader.load_state_dict(resume['loader_state']['test'])
    restore_rng_state(resume.get('rng_state', resume))

training_state.global_step = start_step
training_state.tokens_seen = tokens_seen
training_state.epoch = train_loader.epoch

def save_notebook_checkpoint(path, step, test_loss=None):
    path.parent.mkdir(parents=True, exist_ok=True)
    checkpoint = {
        'architecture': ARCHITECTURE,
        'model_config': model_config,
        'training_config': asdict(training_config),
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scaler_state': scaler.state_dict() if use_scaler else None,
        'step': step,
        'global_step': step,
        'tokens_seen': tokens_seen,
        'global_train_tokens_seen': tokens_seen,
        'epoch': train_loader.epoch,
        'tokens_per_epoch': TOKENS_PER_EPOCH,
        'fractional_epoch': tokens_seen / TOKENS_PER_EPOCH,
        'training_state': training_state.state_dict(),
        'history': history,
        'best_validation_step': best_validation_step,
        'test_loss': test_loss,
        'test_model_selection': 'best_validation' if test_loss is not None else None,
        'data_manifest': artifacts.manifest,
        'data_contract': artifacts.contract,
        'tokenizer': tokenizer.to_state(),
        'flops': flops.as_dict(),
        'kv_cache': kv.as_dict(),
        'loader_state': {
            'train': train_loader.state_dict(),
            'validation': validation_loader.state_dict(),
            'test': test_loader.state_dict(),
        },
        'rng_state': {
            'torch': torch.get_rng_state(),
            'cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
            'python': random.getstate(),
            'numpy': np.random.get_state(),
        },
    }
    torch.save(checkpoint, path)
    print('saved:', path)

## 8. Training loop — cell quan trọng nhất

Đọc chậm từng phần:

1. Lấy `GRADIENT_ACCUMULATION_STEPS` microbatches.
2. Forward với `use_cache=False`.
3. Tính causal cross-entropy.
4. Chia tổng loss cho tổng số target tokens của cả optimizer update.
5. Tính LR theo optimizer step, clip gradient, rồi update optimizer.
6. Định kỳ evaluate, lưu history và checkpoint.

In [ ]:
start_time = time.perf_counter()
train_iterator = iter(train_loader)
last_validation_loss = None
if device.type == 'cuda':
    torch.cuda.reset_peak_memory_stats(device)
progress = tqdm(
    range(start_step + 1, MAX_STEPS + 1),
    total=max(0, MAX_STEPS - start_step),
    desc=f'Train {ARCHITECTURE}',
    unit='step',
    dynamic_ncols=True,
)
for step in progress:
    model.train()
    learning_rate = cosine_lr(
        step,
        warmup_steps=WARMUP_STEPS,
        max_steps=MAX_STEPS,
        lr=LEARNING_RATE,
        min_lr=LEARNING_RATE * MIN_LR_RATIO,
    )
    set_optimizer_lr(optimizer, learning_rate)
    optimizer.zero_grad(set_to_none=True)
    microbatches = []
    for _ in range(GRADIENT_ACCUMULATION_STEPS):
        try:
            inputs, targets = next(train_iterator)
        except StopIteration:
            train_iterator = iter(train_loader)
            inputs, targets = next(train_iterator)
        microbatches.append((inputs, targets))
    total_tokens = sum(targets.numel() for _, targets in microbatches)
    step_loss = 0.0

    for inputs, targets in microbatches:
        inputs = inputs.to(device)
        targets = targets.to(device)
        autocast_context = (
            torch.autocast(device_type='cuda', dtype=amp_dtype)
            if use_amp else nullcontext()
        )
        with autocast_context:
            logits = model(inputs, use_cache=False)
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)), targets.reshape(-1), reduction='sum'
            )
        step_loss += float(loss.detach())
        scaled_loss = loss / total_tokens
        if use_scaler:
            scaler.scale(scaled_loss).backward()
        else:
            scaled_loss.backward()
    tokens_seen += total_tokens
    training_state.global_step = step
    training_state.tokens_seen = tokens_seen
    training_state.epoch = train_loader.epoch

    if use_scaler:
        scaler.unscale_(optimizer)
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
    if use_scaler:
        scaler.step(optimizer)
        scaler.update()
    else:
        optimizer.step()

    elapsed = max(time.perf_counter() - start_time, 1e-9)
    tokens_per_second = tokens_seen / elapsed
    peak_vram_gb = (
        torch.cuda.max_memory_allocated(device) / (1024 ** 3)
        if device.type == 'cuda' else 0.0
    )

    should_eval = step == 1 or step % EVAL_EVERY_STEPS == 0 or step == MAX_STEPS
    if should_eval:
        train_stats = evaluate_loader(train_loader, EVAL_BATCHES_DURING_TRAINING)
        validation_stats = evaluate_loader(validation_loader, EVAL_BATCHES_DURING_TRAINING)
        last_validation_loss = float(validation_stats['loss'])
        record = {
            'step': step,
            'global_step': step,
            'tokens_seen': tokens_seen,
            'epoch': train_loader.epoch,
            'fractional_epoch': tokens_seen / TOKENS_PER_EPOCH,
            'train_loss': train_stats['loss'],
            'validation_loss': last_validation_loss,
            'validation_perplexity': math.exp(min(last_validation_loss, 20.0)),
            'learning_rate': learning_rate,
            'micro_train_loss': step_loss / total_tokens,
            'grad_norm': float(grad_norm),
            'tokens_per_second': tokens_per_second,
            'peak_vram_gb': peak_vram_gb,
            'estimated_training_flops': tokens_seen * flops.training_flops_per_token,
            'elapsed_seconds': elapsed,
        }
        history.append(record)
        if last_validation_loss < best_validation_loss:
            best_validation_loss = last_validation_loss
            best_validation_step = step
            torch.save(model.state_dict(), best_model_path)

    postfix = {
        'loss': f'{step_loss / total_tokens:.4f}',
        'lr': f'{learning_rate:.2e}',
        'grad': f'{float(grad_norm):.2f}',
        'tok/s': f'{tokens_per_second:.0f}',
        'epoch': f'{tokens_seen / TOKENS_PER_EPOCH:.2f}',
    }
    if last_validation_loss is not None:
        postfix['val'] = f'{last_validation_loss:.4f}'
    if device.type == 'cuda':
        postfix['vram'] = f'{peak_vram_gb:.1f}G'
    progress.set_postfix(postfix)

    if step % SAVE_EVERY_STEPS == 0 or step == MAX_STEPS:
        save_notebook_checkpoint(RUN_DIR / f'checkpoint_step_{step}.pt', step)

progress.close()
save_notebook_checkpoint(RUN_DIR / 'checkpoint.pt', MAX_STEPS)

## 9. Final evaluation trên toàn bộ held-out data

Đây mới là số dùng cho report cuối. Không dùng training loss để kết luận architecture tốt hơn; dùng validation/test loss trên cùng fixed token streams.

In [ ]:
# Report held-out metrics for the best validation checkpoint.
final_state = {name: value.detach().clone() for name, value in model.state_dict().items()}
if best_model_path.exists():
    model.load_state_dict(torch.load(best_model_path, map_location=device, weights_only=True))
final_validation = evaluate_loader(validation_loader, max_batches=None)
final_test = evaluate_loader(test_loader, max_batches=None)
model.load_state_dict(final_state)
final_metrics = {
    'architecture': ARCHITECTURE,
    'parameters': parameter_count(model),
    'active_parameters': active_parameter_count(model, ARCHITECTURE),
    'validation_loss': final_validation['loss'],
    'validation_perplexity': math.exp(min(float(final_validation['loss']), 20.0)),
    'test_loss': final_test['loss'],
    'test_perplexity': math.exp(min(float(final_test['loss']), 20.0)),
    'validation_tokens': final_validation['tokens'],
    'test_tokens': final_test['tokens'],
    'training_flops': tokens_seen * flops.training_flops_per_token,
    'tokens_seen': tokens_seen,
    'fractional_epoch': tokens_seen / TOKENS_PER_EPOCH,
    'elapsed_seconds': history[-1].get('elapsed_seconds') if history else None,
    'tokens_per_second': history[-1].get('tokens_per_second') if history else None,
    'peak_vram_gb': max((float(row.get('peak_vram_gb', 0.0)) for row in history), default=0.0),
    'kv_cache_bytes_at_context': kv.total_bytes,
    'best_validation_step': best_validation_step,
    'model_selection': 'best_validation',
}
print(json.dumps(final_metrics, indent=2))
(RUN_DIR / 'final_metrics.json').write_text(json.dumps(final_metrics, indent=2), encoding='utf-8')
(RUN_DIR / 'history.json').write_text(json.dumps(history, indent=2), encoding='utf-8')
(RUN_DIR / 'run_config.json').write_text(json.dumps({
    'architecture': ARCHITECTURE,
    'model': model_config,
    'training': asdict(training_config),
    'data': {
        'artifact_name': ARTIFACT_NAME,
        'artifact_manifest': str(MANIFEST_PATH),
        'train_tokens': len(train_ids),
        'validation_tokens': len(validation_ids),
        'test_tokens': len(test_ids),
        'tokens_per_update': TOKENS_PER_UPDATE,
        'tokens_per_epoch': TOKENS_PER_EPOCH,
    },
    'device': str(device),
}, indent=2), encoding='utf-8')
save_notebook_checkpoint(RUN_DIR / 'checkpoint.pt', MAX_STEPS, test_loss=final_test['loss'])

In [ ]:
steps = np.asarray([row['step'] for row in history], dtype=float)
tokens_curve = np.asarray([row['tokens_seen'] for row in history], dtype=float)
train_curve = np.asarray([row['train_loss'] for row in history], dtype=float)
validation_curve = np.asarray([row['validation_loss'] for row in history], dtype=float)
validation_ppl = np.exp(np.minimum(validation_curve, 20.0))
learning_rates = np.asarray([row.get('learning_rate', np.nan) for row in history], dtype=float)
grad_norms = np.asarray([row.get('grad_norm', np.nan) for row in history], dtype=float)
tokens_per_second = np.asarray([row.get('tokens_per_second', np.nan) for row in history], dtype=float)
peak_vram = np.asarray([row.get('peak_vram_gb', np.nan) for row in history], dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(tokens_curve, train_curve, marker='o', label='train loss')
axes[0].plot(tokens_curve, validation_curve, marker='o', label='validation loss')
axes[0].set(xlabel='tokens seen', ylabel='cross-entropy loss', title=f'{ARCHITECTURE}: loss vs tokens')
axes[0].grid(True, alpha=0.3)
axes[0].legend()
axes[1].plot(tokens_curve, validation_ppl, marker='o', color='tab:orange')
axes[1].set(xlabel='tokens seen', ylabel='perplexity', title=f'{ARCHITECTURE}: validation perplexity')
axes[1].grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(RUN_DIR / 'loss_curves.png', dpi=160, bbox_inches='tight')
plt.show()
plt.close(fig)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].plot(steps, learning_rates, marker='o')
axes[0, 0].set(title='learning rate', xlabel='step', ylabel='lr')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 1].plot(steps, grad_norms, marker='o', color='tab:red')
axes[0, 1].set(title='gradient norm', xlabel='step', ylabel='norm')
axes[0, 1].grid(True, alpha=0.3)
axes[1, 0].plot(tokens_curve, tokens_per_second, marker='o', color='tab:green')
axes[1, 0].set(title='throughput', xlabel='tokens seen', ylabel='tokens/sec')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 1].plot(steps, peak_vram, marker='o', color='tab:purple')
axes[1, 1].set(title='peak VRAM', xlabel='step', ylabel='GB')
axes[1, 1].grid(True, alpha=0.3)
fig.suptitle(f'{ARCHITECTURE}: training diagnostics')
fig.tight_layout()
fig.savefig(RUN_DIR / 'training_diagnostics.png', dpi=160, bbox_inches='tight')
plt.show()
plt.close(fig)

plot_paths = save_training_plots(history, RUN_DIR / 'plots', title=f'{ARCHITECTURE} training')
print('Saved standardized plots:', [str(path) for path in plot_paths])

## 10. Generate text và đo accumulated KV cache

KV cache không dùng trong pretraining. Nó được dùng khi generate autoregressively để không tính lại K/V của toàn bộ prefix ở mỗi token.

In [ ]:
prompt_text = 'The small model learns'
prompt_ids = torch.tensor([tokenizer.encode(prompt_text)], dtype=torch.long, device=device)
NEW_TOKENS = 64

def decode(ids):
    return tokenizer.decode(ids[0].detach().cpu().tolist())

generation_benchmarks = []
for use_cache in (True, False):
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats(device)
    synchronize(device)
    start = time.perf_counter()
    generated = run_generation(model, prompt_ids, NEW_TOKENS, use_cache)
    synchronize(device)
    elapsed = time.perf_counter() - start
    cache_stats = collect_kv_cache_stats(model) if use_cache else {'bytes': 0, 'tokens': 0, 'bytes_per_token': 0.0}
    benchmark = {
        'use_cache': use_cache,
        'seconds': elapsed,
        'tokens_per_second': NEW_TOKENS / elapsed,
        'actual_kv_cache': cache_stats,
        'peak_vram_bytes': torch.cuda.max_memory_allocated(device) if device.type == 'cuda' else 0,
    }
    generation_benchmarks.append(benchmark)
    print(json.dumps(benchmark, indent=2))
    if use_cache:
        cached_output = generated
    else:
        uncached_output = generated

generated_text = decode(cached_output)
print('Cached/uncached equal:', torch.equal(cached_output, uncached_output))
print(generated_text)
(RUN_DIR / 'inference_benchmark.json').write_text(json.dumps(generation_benchmarks, indent=2), encoding='utf-8')
(RUN_DIR / 'generation.txt').write_text(f'Prompt: {prompt_text}\n\n{generated_text}\n', encoding='utf-8')
print('Saved inference artifacts:', RUN_DIR / 'inference_benchmark.json', RUN_DIR / 'generation.txt')

## 11. Optional: xem profile các architecture trước khi train nhiều model

Cell này chỉ build model và tính profile, chưa train thêm. Với MoE, hãy nhìn cả total parameters lẫn active parameters/FLOPs.

In [ ]:
profiles = []
for name in ('mha', 'gqa', 'mla', 'moe', 'v4'):
    cfg = {**base_model_config, **architecture_overrides.get(name, {})}
    candidate = build_model(name, cfg).to(device)
    candidate_flops = estimate_flops(candidate, name)
    candidate_kv = estimate_kv_cache(candidate, name, CONTEXT_LENGTH)
    profiles.append({
        'architecture': name,
        'total_parameters': parameter_count(candidate),
        'active_parameters': active_parameter_count(candidate, name),
        'forward_flops_per_token': candidate_flops.forward_flops_per_token,
        'kv_cache_bytes_per_token': candidate_kv.bytes_per_token,
    })
    del candidate
print(json.dumps(profiles, indent=2))
(RUN_DIR / 'model_profiles.json').write_text(json.dumps(profiles, indent=2), encoding='utf-8')
profile_names = [row['architecture'] for row in profiles]
total_params = np.asarray([row['total_parameters'] for row in profiles], dtype=float) / 1e6
active_params = np.asarray([row['active_parameters'] for row in profiles], dtype=float) / 1e6
forward_flops = np.asarray([row['forward_flops_per_token'] for row in profiles], dtype=float) / 1e9
kv_bytes = np.asarray([row['kv_cache_bytes_per_token'] for row in profiles], dtype=float) / 1024

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].bar(profile_names, total_params, label='total')
axes[0].bar(profile_names, active_params, alpha=0.7, label='active')
axes[0].set(title='parameter profile', ylabel='parameters (M)')
axes[0].legend()
axes[1].bar(profile_names, forward_flops, color='tab:orange')
axes[1].set(title='compute profile', ylabel='FLOPs/token (B)')
axes[2].bar(profile_names, kv_bytes, color='tab:green')
axes[2].set(title='KV-cache profile', ylabel='bytes/token (KB)')
for axis in axes:
    axis.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(RUN_DIR / 'model_profile.png', dpi=160, bbox_inches='tight')
plt.show()
plt.close(fig)

## Kết luận cần ghi vào report

Mỗi run nên lưu ít nhất:

```text
architecture
total parameters
active parameters
training tokens
validation/test loss và perplexity
training FLOPs
tokens/sec
peak VRAM
accumulated KV-cache bytes

Run folder artifacts:
- checkpoint_step_*.pt: intermediate checkpoints, saved every 1500 steps
- best_model.pt and checkpoint.pt: best-validation and final resumable states
- final_metrics.json, history.json, run_config.json
- inference_benchmark.json, generation.txt, model_profiles.json
- loss_curves.png, training_diagnostics.png, model_profile.png
```

Biểu đồ quan trọng nhất cho architecture research là `validation loss vs tokens` hoặc `validation loss vs training FLOPs`, không chỉ loss cuối cùng.